# Lesson 4: Database Design

**Week 2 · Data Engineering Course**

---

**Prerequisites:** Lab 2 complete, Lessons 1–3 read.

**What you will learn:**
- Primary keys — SERIAL and BIGSERIAL for auto-increment
- Foreign keys — enforcing relationships between tables
- Relationships — one-to-many, many-to-many
- Normalisation — 1NF, 2NF, 3NF
- Indexes — when and how to create them
- EXPLAIN — seeing how PostgreSQL executes a query

**All SQL runs in pgAdmin → Tools → Query Tool → select `de_course`**

---

## 4.1 Primary Keys

A primary key uniquely identifies every row. No two rows can share the same primary key, and it can never be NULL.

**In Week 1, you used INTEGER primary keys and loaded IDs from the CSV.** In production, databases generate primary key values automatically so no application code needs to track the next available ID.

### SERIAL and BIGSERIAL

`SERIAL` is shorthand for an auto-incrementing integer primary key:

```sql
CREATE TABLE regions (
    region_id  SERIAL       PRIMARY KEY,
    region_name VARCHAR(50) NOT NULL
);
```

Every INSERT that omits `region_id` gets the next available number automatically:

```sql
INSERT INTO regions (region_name) VALUES ('North');
INSERT INTO regions (region_name) VALUES ('South');
-- region_id values are 1 and 2 automatically
```

| Type | Range | Use when |
|------|-------|----------|
| `SERIAL` | 1 to 2,147,483,647 | Tables with fewer than ~2 billion rows |
| `BIGSERIAL` | 1 to 9.2 quintillion | Large tables — use by default if unsure |

> Modern PostgreSQL also supports `GENERATED ALWAYS AS IDENTITY` which is the SQL-standard equivalent. Both are fine; SERIAL is more common in tutorials.

---

## 4.2 Foreign Keys

A foreign key is a column in one table whose values must match the primary key of another table. It enforces **referential integrity** — you cannot have an order for a customer that does not exist.

**Syntax:**

```sql
creating_column data_type REFERENCES other_table(other_column)
```

**Example — orders referencing customers:**

```sql
CREATE TABLE orders (
    order_id       SERIAL   PRIMARY KEY,
    customer_id    INTEGER  REFERENCES customers(customer_id),
    order_date     DATE,
    status         VARCHAR(20)
);
```

**What the database enforces:**
- You cannot INSERT an order with a `customer_id` that does not exist in `customers`
- You cannot DELETE a customer who has existing orders (by default)

```sql
-- This will fail if customer_id 999 does not exist:
INSERT INTO orders (customer_id, order_date, status)
VALUES (999, '2024-01-01', 'pending');
-- ERROR: insert or update on table orders violates foreign key constraint
```

---

### ON DELETE behaviour

What happens when you delete a referenced row? You choose:

| Option | Behaviour |
|--------|-----------|
| `RESTRICT` (default) | Prevent the delete — error |
| `CASCADE` | Delete all child rows automatically |
| `SET NULL` | Set the foreign key column to NULL |
| `SET DEFAULT` | Set the foreign key column to its default value |

```sql
CREATE TABLE orders (
    order_id       SERIAL   PRIMARY KEY,
    customer_id    INTEGER  REFERENCES customers(customer_id) ON DELETE CASCADE,
    order_date     DATE,
    status         VARCHAR(20)
);
```

With `ON DELETE CASCADE`: deleting customer_id 1 also deletes all their orders and — if order_items also has CASCADE — all their line items.

**Choose carefully.** CASCADE feels convenient but can silently delete large amounts of data. In analytical databases, `RESTRICT` (the default) is usually safer.

---

## 4.3 Relationships

### One-to-many

The most common relationship. One customer has many orders. One order has many order items.

```
customers (1) ──< orders (many) ──< order_items (many)
```

The foreign key always lives on the **many** side (orders.customer_id, order_items.order_id).

### Many-to-many

A student can enroll in many courses; a course can have many students. Neither side holds a foreign key to the other — instead a **junction table** holds both:

```sql
CREATE TABLE enrollments (
    student_id  INTEGER REFERENCES students(student_id),
    course_id   INTEGER REFERENCES courses(course_id),
    enrolled_at DATE,
    PRIMARY KEY (student_id, course_id)   -- composite primary key
);
```

`order_items` is itself a junction table: a many-to-many between `orders` and `products`, with extra columns (`quantity`, `unit_price`) to describe the relationship.

---

## 4.4 Normalisation

Normalisation is the process of organising a database to reduce redundancy and prevent anomalies (contradictory data, update problems, orphaned records).

### First Normal Form (1NF)

Every cell must hold a single atomic value. No lists or arrays in a column.

```
Bad:  orders | customer_name | products
      1      | Alice         | Laptop, Mouse, Keyboard   ← list in one cell

Good: use order_items with one product per row
```

### Second Normal Form (2NF)

Every non-key column must depend on the **whole** primary key, not just part of it. Relevant when the primary key is composite.

```
Bad:  order_items | order_id | product_id | product_name | quantity
      product_name depends only on product_id, not on (order_id, product_id)

Good: put product_name in a separate products table
```

### Third Normal Form (3NF)

Every non-key column must depend on the key directly, not through another non-key column.

```
Bad:  orders | order_id | customer_id | customer_city
      customer_city depends on customer_id, not directly on order_id

Good: put customer_city in the customers table
```

> The retail database you built is already in 3NF. The rules above explain why each piece of data lives where it does.

---

## 4.5 Indexes

An index is a data structure that lets PostgreSQL find rows much faster — similar to an index in a book. Without an index, PostgreSQL reads every row (a **sequential scan**). With an index, it jumps directly to the matching rows (an **index scan**).

**Creating an index:**

```sql
CREATE INDEX idx_orders_customer_id ON orders(customer_id);
CREATE INDEX idx_orders_order_date  ON orders(order_date);
CREATE INDEX idx_order_items_order_id ON order_items(order_id);
```

**When to create an index:**
- Columns used in WHERE clauses frequently
- Columns used in JOIN conditions (foreign keys)
- Columns used in ORDER BY on large tables

**PostgreSQL creates indexes automatically for:**
- PRIMARY KEY columns
- UNIQUE constraints

**Indexes have a cost:** they slow down INSERT, UPDATE, and DELETE because the index must be updated too. Do not add indexes everywhere — add them where queries are actually slow.

**Composite indexes** cover multiple columns:

```sql
CREATE INDEX idx_orders_customer_date ON orders(customer_id, order_date);
```

This helps queries that filter on both columns together.

---

## 4.6 EXPLAIN — Reading the Query Plan

`EXPLAIN` shows how PostgreSQL plans to execute a query — without actually running it.

```sql
EXPLAIN
SELECT *
FROM orders
WHERE customer_id = 1;
```

Sample output:
```
Seq Scan on orders  (cost=0.00..1.50 rows=4 width=28)
  Filter: (customer_id = 1)
```

`EXPLAIN ANALYZE` runs the query and shows actual vs estimated rows:

```sql
EXPLAIN ANALYZE
SELECT * FROM orders WHERE customer_id = 1;
```

**What to look for:**

| Plan node | Meaning |
|-----------|---------|
| `Seq Scan` | Reads every row — fine for small tables, slow on large ones |
| `Index Scan` | Uses an index — fast for filtered queries |
| `Hash Join` | Joins using a hash table — common for INNER JOINs |
| `Nested Loop` | Joins by looping — good when one side is small |

At this stage: run EXPLAIN to see whether your query uses an index. You will use it much more in later weeks when optimising pipelines on large datasets.

---

## 4.7 Practical: Inspect the Retail Database Design

Run these queries in pgAdmin to inspect the database you built.

**List all tables:**

```sql
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
```

**Show columns and types for orders:**

```sql
SELECT column_name, data_type, is_nullable
FROM information_schema.columns
WHERE table_name = 'orders'
ORDER BY ordinal_position;
```

**List all foreign key constraints:**

```sql
SELECT
    tc.table_name  AS child_table,
    kcu.column_name AS fk_column,
    ccu.table_name  AS parent_table,
    ccu.column_name AS pk_column
FROM information_schema.table_constraints tc
JOIN information_schema.key_column_usage kcu
    ON tc.constraint_name = kcu.constraint_name
JOIN information_schema.constraint_column_usage ccu
    ON tc.constraint_name = ccu.constraint_name
WHERE tc.constraint_type = 'FOREIGN KEY'
ORDER BY child_table;
```

**List all indexes:**

```sql
SELECT indexname, tablename, indexdef
FROM pg_indexes
WHERE schemaname = 'public'
ORDER BY tablename, indexname;
```

---

## 4.8 Key Takeaways

1. **SERIAL / BIGSERIAL** auto-generates primary key values on every INSERT — no application code needed.
2. **Foreign keys** enforce referential integrity — you cannot insert a child row without a matching parent.
3. `ON DELETE CASCADE` auto-removes child rows when a parent is deleted. Use with care.
4. The foreign key always lives on the **many** side of a one-to-many relationship.
5. **Many-to-many** relationships use a junction table holding two foreign keys.
6. **Normalisation** eliminates redundancy. 3NF means every column depends directly on the primary key.
7. **Indexes** speed up reads but slow down writes. Add them on WHERE and JOIN columns, not everywhere.
8. PostgreSQL automatically indexes PRIMARY KEY and UNIQUE columns.
9. **EXPLAIN** shows the query plan. Look for Seq Scan on large tables as a sign that an index might help.